# UCS420 – Assignment 10
## Diabetes Care Information Assistant
**Assignment track:** Odd roll number

This notebook builds a Python-based healthcare cognitive assistant focused on diabetes care. It covers the assignment requirements: labelled query corpus, hierarchical taxonomy, domain concepts and relationships, NLP preprocessing, TF-IDF, intent classification, evaluation, a diabetes knowledge base, reasoning rules, integrated query handling, and an interactive interface suitable for Google Colab.

> **Important:** This is an educational information assistant, not a diagnostic tool. It does not prescribe or change medication. For urgent symptoms, it directs the user to emergency medical care. Always follow an individual's clinician-provided diabetes care plan.


## 1. Install and import libraries

Run this cell first in Google Colab. NLTK resources are downloaded here so preprocessing works in a fresh runtime.

In [ ]:
!pip -q install nltk scikit-learn pandas matplotlib ipywidgets

import re
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from IPython.display import display, Markdown

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("Libraries and NLP resources are ready.")

## 2. Hierarchical taxonomy

The assistant uses six intent categories grouped under four major diabetes-care areas. Each major category has at least two subcategories.

In [ ]:
taxonomy = {
    "Diabetes Basics": {
        "understanding_diabetes": "Meaning, types and general understanding of diabetes",
        "risk_and_prevention": "Risk factors, prevention and screening"
    },
    "Monitoring and Treatment": {
        "glucose_monitoring": "Blood glucose checks, targets and glucose records",
        "medication": "General information about prescribed diabetes medicines and insulin safety"
    },
    "Daily Diabetes Management": {
        "diet_nutrition": "Food choices, meal planning and carbohydrate awareness",
        "exercise_lifestyle": "Physical activity, sleep, hydration and daily habits"
    },
    "Health and Safety": {
        "symptoms_complications": "Common symptoms and possible long-term complications",
        "emergency_support": "Urgent warning signs and emergency escalation"
    }
}

for major, subcategories in taxonomy.items():
    print(f"\n{major}")
    for intent, description in subcategories.items():
        print(f"  • {intent}: {description}")

## 3. Labelled corpus

The corpus below contains more than 40 example queries across eight intents. The examples are short, varied user utterances intended for a classroom intent-classification exercise. They are not clinical advice.

In [ ]:
examples = {
    "understanding_diabetes": [
        "What is diabetes?",
        "Explain type 1 diabetes in simple words",
        "What happens in the body when someone has diabetes?",
        "How is type 2 diabetes different from type 1?",
        "Can you explain insulin resistance?",
        "Why does blood sugar become high in diabetes?",
        "Is diabetes a long term condition?",
        "Tell me the basic meaning of blood glucose"
    ],
    "risk_and_prevention": [
        "What are the risk factors for type 2 diabetes?",
        "Can diabetes be prevented?",
        "How does family history affect diabetes risk?",
        "Who should get screened for diabetes?",
        "Does being physically inactive increase diabetes risk?",
        "What lifestyle habits can lower diabetes risk?",
        "What is prediabetes?",
        "How can someone reduce their chance of developing type 2 diabetes?"
    ],
    "glucose_monitoring": [
        "How do I check my blood sugar at home?",
        "When should I monitor my glucose?",
        "How does a glucometer work?",
        "What is a continuous glucose monitor?",
        "How should I record my glucose readings?",
        "Why is it useful to track blood sugar levels?",
        "What does fasting blood glucose mean?",
        "How can I keep a log of my sugar readings?"
    ],
    "medication": [
        "What is metformin generally used for?",
        "Why might a doctor prescribe insulin?",
        "What should I know about storing insulin?",
        "Can I stop my diabetes medicine when I feel better?",
        "Why is it important to take prescribed medicine as directed?",
        "What are common forms of diabetes medication?",
        "I forgot a dose of my diabetes medicine, what should I do?",
        "Can you explain the difference between insulin and tablets?"
    ],
    "diet_nutrition": [
        "What foods are suitable for someone with diabetes?",
        "How can I plan balanced meals?",
        "Why should I pay attention to carbohydrates?",
        "Are whole grains a useful food choice?",
        "What are some healthy snack ideas for diabetes?",
        "How can portion size affect blood sugar?",
        "Should I avoid all fruits if I have diabetes?",
        "What does a balanced diabetes-friendly plate look like?"
    ],
    "exercise_lifestyle": [
        "What kind of exercise is helpful for diabetes?",
        "How can walking support diabetes management?",
        "Why is sleep important for blood sugar control?",
        "What daily habits support diabetes care?",
        "How should I prepare for physical activity with diabetes?",
        "Can stress affect blood glucose?",
        "How can I build a regular exercise routine?",
        "Why is hydration important?"
    ],
    "symptoms_complications": [
        "What are common symptoms of diabetes?",
        "Why do people with diabetes feel very thirsty?",
        "Can diabetes affect eyesight?",
        "What is diabetic neuropathy?",
        "Why can diabetes affect the kidneys?",
        "What are signs of high blood sugar?",
        "What are signs of low blood sugar?",
        "Can diabetes affect foot health?"
    ],
    "emergency_support": [
        "I am confused and fainting and my blood sugar is very low",
        "Someone with diabetes is unconscious",
        "I have severe difficulty breathing and diabetes",
        "A person with diabetes is having a seizure",
        "I am vomiting repeatedly and feel extremely weak with diabetes",
        "My family member with diabetes cannot be woken up",
        "I have severe chest pain and diabetes",
        "There is a medical emergency involving a diabetic person"
    ]
}

records = [
    {"query": query, "intent": intent}
    for intent, queries in examples.items()
    for query in queries
]
corpus = pd.DataFrame(records)
print("Number of labelled queries:", len(corpus))
print("Number of intent categories:", corpus["intent"].nunique())
display(corpus.groupby("intent").size().rename("query_count").to_frame())
display(corpus.head(10))

## 4. Domain concepts and relationships

The knowledge model below lists more than eight diabetes-domain concepts and at least ten relationships. These relationships represent a simplified educational domain map, not a complete clinical ontology.

In [ ]:
concepts = {
    "Diabetes": "A group of conditions involving elevated blood glucose",
    "Blood glucose": "The amount of glucose in the blood",
    "Insulin": "A hormone involved in regulating blood glucose",
    "Pancreas": "An organ that produces insulin",
    "Glucose meter": "A device used to measure blood glucose",
    "Continuous glucose monitor": "A device that tracks glucose trends over time",
    "Medication": "A treatment prescribed by a qualified healthcare professional",
    "Nutrition": "Food and carbohydrate choices relevant to diabetes care",
    "Physical activity": "Movement and exercise as part of a healthy lifestyle",
    "Hypoglycemia": "Blood glucose that is lower than the person's safe range",
    "Hyperglycemia": "Blood glucose that is higher than the person's safe range",
    "Healthcare professional": "A qualified clinician involved in care"
}

relationships = [
    ("Diabetes", "is associated with", "Blood glucose"),
    ("Pancreas", "produces", "Insulin"),
    ("Insulin", "helps regulate", "Blood glucose"),
    ("Glucose meter", "measures", "Blood glucose"),
    ("Continuous glucose monitor", "tracks", "Blood glucose"),
    ("Medication", "may be prescribed to manage", "Diabetes"),
    ("Nutrition", "can influence", "Blood glucose"),
    ("Physical activity", "can influence", "Blood glucose"),
    ("Hypoglycemia", "means low", "Blood glucose"),
    ("Hyperglycemia", "means high", "Blood glucose"),
    ("Healthcare professional", "guides use of", "Medication"),
    ("Healthcare professional", "supports management of", "Diabetes")
]

print("Number of concepts:", len(concepts))
for name, definition in concepts.items():
    print(f"• {name}: {definition}")

print("\nRelationships:")
for subject, relation, obj in relationships:
    print(f"• {subject} — {relation} → {obj}")
print("\nNumber of relationships:", len(relationships))

## 5. NLP preprocessing

The preprocessing pipeline performs all four required operations:
1. Lowercase conversion
2. Tokenization
3. Stopword removal
4. Lemmatization

The function is used by the TF-IDF vectorizer as its tokenizer/preprocessor.

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    tokens = re.findall(r"[a-z]+", text)
    tokens = [token for token in tokens if token not in stop_words]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return tokens

sample_query = "How can I monitor my blood glucose levels?"
print("Original:", sample_query)
print("Processed tokens:", preprocess_text(sample_query))

## 6. TF-IDF feature extraction and intent classifier

A stratified train/test split is used so each intent is represented in both sets. TF-IDF converts text into numerical features, and Logistic Regression learns to classify the intent.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    corpus["query"],
    corpus["intent"],
    test_size=0.25,
    random_state=42,
    stratify=corpus["intent"]
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        tokenizer=preprocess_text,
        token_pattern=None,
        lowercase=False,
        ngram_range=(1, 2),
        min_df=1
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)
print("Training examples:", len(X_train))
print("Testing examples:", len(X_test))
print("Classifier trained successfully.")

## 7. Model evaluation

The assignment asks for accuracy, precision, recall, F1-score and a confusion matrix. Macro averages give each intent equal weight.

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision (macro)", "Recall (macro)", "F1-score (macro)"],
    "Score": [accuracy, precision, recall, f1]
})
display(metrics)

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

labels = sorted(corpus["intent"].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(11, 8))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(
    ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False
)
plt.title("Confusion Matrix – Diabetes Care Intent Classifier")
plt.tight_layout()
plt.show()

## 8. Diabetes-specific knowledge base

Each supported intent maps to a short educational response. The content is deliberately general. It does not provide individualized glucose targets, medication doses, or treatment changes.

In [ ]:
knowledge_base = {
    "understanding_diabetes": (
        "Diabetes is a condition in which blood glucose is higher than the healthy range. "
        "The main types include type 1 and type 2 diabetes, which have different causes and care needs. "
        "A healthcare professional can explain an individual's diagnosis and care plan."
    ),
    "risk_and_prevention": (
        "Type 2 diabetes risk can be influenced by family history, age, physical activity, and other factors. "
        "Balanced eating, regular activity, and appropriate screening can support prevention or early detection. "
        "A clinician can advise whether screening is appropriate."
    ),
    "glucose_monitoring": (
        "Blood glucose may be checked with a glucose meter or monitored using a continuous glucose monitor. "
        "When and how often to check depends on the person's treatment plan. Record readings and discuss patterns "
        "with the diabetes care team rather than changing treatment based on one reading."
    ),
    "medication": (
        "Diabetes medicines include different types of tablets and injectable treatments, including insulin. "
        "Use medicines only as prescribed. Do not stop, start, or change a dose based on this assistant. "
        "For a missed dose or side effects, check the medicine instructions and contact a pharmacist or clinician."
    ),
    "diet_nutrition": (
        "A balanced eating pattern often includes non-starchy vegetables, suitable portions of whole grains or other "
        "high-fibre carbohydrates, protein sources, and healthy fats. Carbohydrate amount and meal needs vary by person. "
        "A registered dietitian or diabetes care team can help create an individualized plan."
    ),
    "exercise_lifestyle": (
        "Regular physical activity, adequate sleep, hydration, and stress management can support general health "
        "and diabetes management. Choose activity that is appropriate for your health and follow your care team's "
        "advice about checking glucose and preventing low blood sugar during exercise."
    ),
    "symptoms_complications": (
        "Possible diabetes symptoms include increased thirst, frequent urination, tiredness, and blurred vision. "
        "Over time, diabetes can affect the eyes, kidneys, nerves, heart, and feet. New, worsening, or concerning "
        "symptoms should be discussed with a healthcare professional."
    ),
    "emergency_support": (
        "This may be an emergency. Contact your local emergency number or go to the nearest emergency department now. "
        "If the person is unconscious, having a seizure, severely confused, or unable to swallow, do not give food "
        "or drink by mouth. Follow emergency dispatcher instructions and the person's clinician-provided emergency plan."
    )
}

print("Knowledge base categories:", len(knowledge_base))
for key in knowledge_base:
    print("•", key)

## 9. Reasoning rules

The assistant applies explicit rules before returning a response:
- **Emergency escalation:** urgent warning phrases bypass normal classification.
- **Low-confidence handling:** uncertain model predictions are not treated as reliable.
- **Unsupported-query handling:** queries outside the diabetes assistant's scope are redirected.
- **Medication safety:** the assistant will not recommend changing or stopping medication.
- **No diagnosis/prescription:** it does not diagnose or prescribe treatment.
- **General-information response:** supported intents retrieve information from the knowledge base.

The confidence threshold is a simple classroom heuristic and should not be interpreted as a clinically validated safety threshold.

In [ ]:
EMERGENCY_TERMS = [
    "unconscious", "cannot be woken", "can't wake", "seizure",
    "fainting", "severe chest pain", "difficulty breathing",
    "unable to breathe", "not responding", "severely confused"
]

MEDICATION_CHANGE_TERMS = [
    "change my dose", "increase my dose", "double my dose",
    "stop taking", "stop my medicine", "take extra insulin",
    "how much insulin should i take", "what dose should i take"
]

DIABETES_TERMS = [
    "diabetes", "diabetic", "blood sugar", "blood glucose",
    "insulin", "glucose", "metformin", "hypoglycemia",
    "hyperglycemia", "sugar level", "prediabetes"
]

CONFIDENCE_THRESHOLD = 0.30

def contains_any(text, terms):
    return any(term in text for term in terms)

def assistant_response(query):
    clean_query = query.strip()
    lowered = clean_query.lower()

    if not clean_query:
        return {
            "intent": "no_query",
            "confidence": 0.0,
            "response": "Please enter a question about diabetes care."
        }

    # Rule 1: Emergency escalation always takes priority over the classifier.
    if contains_any(lowered, EMERGENCY_TERMS):
        return {
            "intent": "emergency_support",
            "confidence": 1.0,
            "response": knowledge_base["emergency_support"]
        }

    # Rule 2: Do not advise on medication dose changes.
    if contains_any(lowered, MEDICATION_CHANGE_TERMS):
        return {
            "intent": "medication_safety_rule",
            "confidence": 1.0,
            "response": (
                "I cannot recommend changing, stopping, or calculating a medication or insulin dose. "
                "Follow the prescription and contact your doctor or pharmacist for individualized advice. "
                "If you may have taken an incorrect dose and feel unwell, seek urgent medical help."
            )
        }

    # Rule 3: Keep the assistant within its diabetes-care scope.
    if not contains_any(lowered, DIABETES_TERMS):
        return {
            "intent": "unsupported_query",
            "confidence": 0.0,
            "response": (
                "I am a Diabetes Care Information Assistant and can answer general questions about diabetes, "
                "glucose monitoring, nutrition, lifestyle, medicines, symptoms, and prevention. "
                "This question appears outside my supported scope."
            )
        }

    probabilities = model.predict_proba([clean_query])[0]
    best_index = int(np.argmax(probabilities))
    predicted_intent = model.classes_[best_index]
    confidence = float(probabilities[best_index])

    # Rule 4: Low-confidence handling.
    if confidence < CONFIDENCE_THRESHOLD:
        return {
            "intent": "low_confidence",
            "confidence": confidence,
            "response": (
                "I am not confident enough to classify that question safely. "
                "Please rephrase it or ask your diabetes care team for advice specific to you."
            )
        }

    # Rule 5: Retrieve the answer from the domain knowledge base.
    return {
        "intent": predicted_intent,
        "confidence": confidence,
        "response": knowledge_base[predicted_intent]
    }

print("Reasoning rules are ready.")

## 10. Test cases required by the assignment

The following tests include a query for each defined intent, a differently worded query, an unsupported query, a low-confidence-style query, and an emergency-related query. Actual classifier confidence depends on the trained model; the low-confidence test is shown with a deliberately unfamiliar diabetes-related phrase and its result is reported rather than assumed.

In [ ]:
test_queries = [
    ("Understanding diabetes", "Can you explain what diabetes means?", "understanding_diabetes"),
    ("Risk and prevention", "How can I lower my risk of type 2 diabetes?", "risk_and_prevention"),
    ("Glucose monitoring", "What device can I use to check glucose at home?", "glucose_monitoring"),
    ("Medication", "Why has my clinician prescribed metformin?", "medication"),
    ("Diet and nutrition", "What should a balanced meal look like for diabetes?", "diet_nutrition"),
    ("Exercise and lifestyle", "Can a daily walk help with diabetes care?", "exercise_lifestyle"),
    ("Symptoms and complications", "Why might a person with diabetes have blurry vision?", "symptoms_complications"),
    ("Emergency", "My diabetic relative is unconscious and cannot be woken up", "emergency_support"),
    ("Different wording", "I want to keep track of my sugar readings", "glucose_monitoring"),
    ("Unsupported", "How do I treat a migraine headache?", "unsupported_query"),
    ("Medication safety", "Should I stop taking my diabetes medicine?", "medication_safety_rule"),
    ("Low-confidence candidate", "Diabetes glucose biomarker oscillation query", "review")
]

test_results = []
for label, query, expected in test_queries:
    result = assistant_response(query)
    test_results.append({
        "Test": label,
        "Query": query,
        "Expected / purpose": expected,
        "Detected intent": result["intent"],
        "Confidence": round(result["confidence"], 3),
        "Response": result["response"]
    })

test_df = pd.DataFrame(test_results)
display(test_df[["Test", "Expected / purpose", "Detected intent", "Confidence"]])
print("\nFull responses:")
for row in test_results:
    print(f"\n[{row['Test']}] {row['Query']}")
    print("Intent:", row["Detected intent"], "| Confidence:", row["Confidence"])
    print(row["Response"])

### 10.1 Inspecting the low-confidence rule

The model's confidence for an unfamiliar query may still be high because a classifier always chooses among its known classes. The explicit threshold handles low-confidence predictions when they occur. The cell below shows the confidence for several diabetes-related, unusual phrasings so the behaviour can be inspected.

In [ ]:
confidence_checks = [
    "diabetes glucose biomarker oscillation query",
    "diabetes metabolic marker fluctuation terminology",
    "diabetes care question about an unfamiliar measurement"
]

for q in confidence_checks:
    result = assistant_response(q)
    print(f"Query: {q}")
    print(f"Intent: {result['intent']} | Confidence: {result['confidence']:.3f}")
    print("-" * 70)

## 11. Interactive assistant interface for Google Colab

This interface accepts multiple queries in one text area (one query per line) and displays a response for each. If widgets are unavailable in a notebook environment, use the simple loop in the next cell.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    query_box = widgets.Textarea(
        value="How do I check my blood sugar at home?\nWhat foods are suitable for someone with diabetes?",
        placeholder="Enter one question per line",
        description="Queries:",
        layout=widgets.Layout(width="100%", height="120px")
    )
    run_button = widgets.Button(
        description="Ask Assistant",
        button_style="primary",
        icon="comment"
    )
    output_area = widgets.Output()

    def on_run_clicked(_):
        with output_area:
            clear_output()
            queries = [q.strip() for q in query_box.value.splitlines() if q.strip()]
            if not queries:
                print("Please enter at least one query.")
                return
            for number, query in enumerate(queries, start=1):
                result = assistant_response(query)
                display(Markdown(
                    f"### Query {number}: {query}\n"
                    f"**Detected intent:** `{result['intent']}`  \n"
                    f"**Confidence:** `{result['confidence']:.3f}`  \n"
                    f"**Assistant:** {result['response']}"
                ))

    run_button.on_click(on_run_clicked)
    display(query_box, run_button, output_area)

except Exception as error:
    print("Interactive widgets could not be loaded:", error)
    print("Use the text-based interface in the next cell.")

In [ ]:
# Text-based fallback: enter several questions, one per line.
# Submit an empty line to finish.
try:
    user_lines = []
    print("Enter diabetes-care questions, one per line. Press Enter on a blank line to finish.")
    while True:
        line = input("You: ").strip()
        if not line:
            break
        user_lines.append(line)

    for i, query in enumerate(user_lines, 1):
        result = assistant_response(query)
        print(f"\nQuery {i}: {query}")
        print(f"Intent: {result['intent']} | Confidence: {result['confidence']:.3f}")
        print("Assistant:", result["response"])
except EOFError:
    print("Interactive input is not available in this environment. Use the Colab widget above.")

## 12. System workflow

1. User enters one or more questions.
2. The assistant checks emergency and medication-safety rules.
3. It checks whether the question is within the diabetes-care domain.
4. NLP preprocessing converts the text into normalized tokens.
5. TF-IDF converts the query into numerical features.
6. Logistic Regression predicts an intent and confidence.
7. A low-confidence result is handled cautiously.
8. The knowledge base returns the relevant educational response.

**Pipeline:** User Query → Safety Rules → Scope Check → NLP Preprocessing → TF-IDF → Intent Classifier → Confidence Check → Knowledge Retrieval → Response



## 13. Conclusion

The Diabetes Care Information Assistant implements the required cognitive-assistant pipeline using a labelled corpus, hierarchical taxonomy, domain concepts and relationships, NLP preprocessing, TF-IDF, Logistic Regression, model evaluation, a domain knowledge base, reasoning rules, and a multi-query interface for Google Colab.

**Limitations:** The dataset is small and synthetic, so evaluation scores are only a classroom demonstration and do not establish real-world clinical reliability. The classifier can misclassify unfamiliar wording. The assistant provides general educational information only; it must not replace professional medical advice, diagnosis, or emergency services.